# Meta-inference pilot: do 14 methods together beat the best one?


**The pilot (instruction 64.34).** Label `dtm_class` (Toxoplasma). Each gene has a held-out call from up to 14 methods. A meta-model (multinomial logistic regression on the one-hot calls) is fitted on 4/5 of the orthogroups and tested on the other 1/5, five times; compared on the same test genes with the best single method (chosen on the training orthogroups), the plurality vote, and the commonest class.

**Stated limit.** The base calls come from the hold-out record's own folds; a strictly nested version would recompute every base method inside each outer fold. This pilot says whether that is worth doing.

Built 2026-11-07.

In [1]:
import os, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from scipy.stats import binomtest
from parascape import track_record as T
L = T.shipped('Tg')
L = L[(L['target'].astype(str) == TARGET) & (L['mode'].astype(str) == 'together')].copy()
first = L.groupby(L['strategy'].astype(str), observed=True)['setting_key'].first()
L = L[L['setting_key'].astype(str).values == L['strategy'].astype(str).map(first).astype(str).values]
calls = L.assign(call=np.where(L['abstained'].astype(bool), 'ABSTAIN', L['prediction'].astype(str))) \
         .pivot_table(index='gene_id', columns='strategy', values='call', aggfunc='first', observed=True).fillna('ABSTAIN')
truth = L.groupby('gene_id', observed=True)['truth'].first().astype(str).reindex(calls.index)
nodes = pd.read_parquet(os.path.join(REPO, 'parascape/data/nodes.parquet')).set_index('gene_id')
groups = nodes['orthogroup'].reindex(calls.index).fillna(pd.Series(calls.index, index=calls.index))
X = pd.get_dummies(calls.astype(str)).astype(float)
print(calls.shape[0], 'genes,', calls.shape[1], 'methods,', truth.nunique(), 'classes')

8137 genes, 14 methods, 4 classes


In [2]:
rows, paired = [], []
for fold, (tr, te) in enumerate(GroupKFold(5).split(X, truth, groups)):
    ytr, yte = truth.iloc[tr], truth.iloc[te]
    meta = LogisticRegression(max_iter=3000, C=1.0).fit(X.iloc[tr], ytr)
    p_meta = meta.predict(X.iloc[te])
    acc_tr = {m: float((calls.iloc[tr][m] == ytr).mean()) for m in calls.columns}
    best = max(acc_tr, key=acc_tr.get)
    p_best = calls.iloc[te][best].to_numpy()
    plural = calls.iloc[te].apply(lambda r: r[r != 'ABSTAIN'].value_counts().index[0] if (r != 'ABSTAIN').any() else 'ABSTAIN', axis=1).to_numpy()
    common = np.full(len(te), ytr.value_counts().index[0])
    y = yte.to_numpy()
    for name, p in (('meta-model', p_meta), (f'best single ({best})', p_best), ('plurality vote', plural), ('commonest class', common)):
        rows.append(dict(fold=fold, method=name.split(' (')[0], chosen=best if name.startswith('best') else '', accuracy=float((p == y).mean()), n=len(y)))
    paired.append(dict(fold=fold, meta_only=int(((p_meta == y) & (p_best != y)).sum()), best_only=int(((p_best == y) & (p_meta != y)).sum())))
res = pd.DataFrame(rows)
summary = res.groupby('method').agg(accuracy=('accuracy', 'mean'), low=('accuracy', 'min'), high=('accuracy', 'max')).round(3).sort_values('accuracy', ascending=False)
pp = pd.DataFrame(paired).sum()
test = binomtest(int(pp.meta_only), int(pp.meta_only + pp.best_only), 0.5)
summary.to_csv(os.path.join(OUT, 'summary.tsv'), sep='\t')
res.to_csv(os.path.join(OUT, 'folds.tsv'), sep='\t', index=False)
print('meta right where the best single is wrong:', int(pp.meta_only), '; the reverse:', int(pp.best_only), '; McNemar p', round(test.pvalue, 4))
summary

meta right where the best single is wrong: 223 ; the reverse: 103 ; McNemar p 0.0


,accuracy,low,high
method,,,
meta-model,0.918,0.899,0.932
best single,0.903,0.888,0.917
plurality vote,0.819,0.801,0.841
commonest class,0.744,0.729,0.757


### Per class, and how well calibrated the meta probabilities are

Out-of-fold meta predictions over all five folds: accuracy within each true class (recall) and the expected calibration error of the meta-model's top probability (10 bins).

In [3]:
oof_pred = pd.Series(index=X.index, dtype=object); oof_prob = pd.Series(index=X.index, dtype=float)
for tr, te in GroupKFold(5).split(X, truth, groups):
    m = LogisticRegression(max_iter=3000, C=1.0).fit(X.iloc[tr], truth.iloc[tr])
    pr = m.predict_proba(X.iloc[te])
    oof_pred.iloc[te] = m.classes_[pr.argmax(axis=1)]; oof_prob.iloc[te] = pr.max(axis=1)
right = (oof_pred == truth)
per_class = pd.DataFrame({'genes': truth.value_counts(), 'recall': right.groupby(truth).mean()}).round(3)
bins = np.clip((oof_prob * 10).astype(int), 0, 9)
ece = float(sum((bins == b).mean() * abs(right[bins == b].mean() - oof_prob[bins == b].mean()) for b in range(10) if (bins == b).any()))
per_class.to_csv(os.path.join(OUT, 'per_class.tsv'), sep='\t')
print('expected calibration error of the meta probabilities:', round(ece, 4))
per_class

expected calibration error of the meta probabilities: 0.0066


,genes,recall
truth,,
GLOB,6053,0.984
SP,676,0.54
SP+TM,266,0.278
TM,1142,0.939
